In [ ]:
!pip install scikit-bio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 96.8 MB/s eta 0:00:00


In [ ]:
import json
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from mpl_toolkits.mplot3d import Axes3D

from scipy import stats
from scipy.stats import ttest_ind
from scipy.stats import mannwhitneyu
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import pdist, squareform

from skbio.stats.distance import DistanceMatrix, permanova

from statsmodels.multivariate.manova import MANOVA
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering, KMeans, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score

from sklearn.ensemble import IsolationForest

from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_curve, roc_auc_score, silhouette_score, r2_score, mean_squared_error
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
df = pd.read_excel("Data_vascular.xlsx")

FileNotFoundError: [Errno 2] No such file or directory: 'Data_vascular.xlsx'

In [ ]:
df

In [ ]:
# Shuffle original dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df

,Permeability,LDL_uptake,Total_ROS,Vascular_Marker,Cell_Signalling,Tube_formation,In_vivo_recovery,Group
0,71,0.30,1.8,68,43,39,35,0
1,68,0.34,1.9,75,47,35,34,0
2,20,1.20,1.0,100,80,100,70,1
3,79,0.19,1.8,70,43,41,28,0
4,17,1.30,0.9,96,88,92,72,1
5,69,2.00,2.2,74,55,44,19,0
6,22,1.40,1.1,96,85,99,79,1
7,23,1.10,0.9,98,78,98,78,1
8,70,0.35,2.1,69,59,46,22,0
9,21,1.80,0.8,92,95,96,82,1


In [ ]:
# Check Duplicate row in dataset
df.duplicated().sum()

np.int64(0)

In [ ]:
df["Group"].value_counts()

,count
Group,
0,7
1,7


In [ ]:
df.describe()

,Permeability,LDL_uptake,Total_ROS,Vascular_Marker,Cell_Signalling,Tube_formation,In_vivo_recovery,Group
count,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000,14.000000
mean,45.857143,0.957143,1.450000,84.714286,68.071429,67.785714,52.142857,0.500000
std,26.720367,0.653210,0.548775,12.418526,20.200778,29.571760,25.806550,0.518875
min,13.000000,0.190000,0.700000,68.000000,43.000000,32.000000,19.000000,0.000000
25%,21.250000,0.310000,0.925000,74.250000,49.250000,40.250000,29.000000,0.000000
50%,47.000000,1.150000,1.500000,85.500000,68.500000,68.000000,52.500000,0.500000
75%,69.750000,1.375000,1.900000,96.000000,87.250000,96.750000,76.500000,1.000000
max,79.000000,2.000000,2.200000,100.000000,95.000000,100.000000,83.000000,1.000000


In [ ]:
features = df.columns

In [ ]:
features

Index(['Permeability', 'LDL_uptake', 'Total_ROS', 'Vascular_Marker',
       'Cell_Signalling', 'Tube_formation', 'In_vivo_recovery', 'Group'],
      dtype='object')

In [ ]:
# Standardization
def standardization(x):
    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(x)
    return x_scaled, scaler

In [ ]:
X = df.drop(columns=["Group"])
y = df["Group"]

# Statistical Modelling


In [ ]:
def statistics(df):
    summary_dict = df.describe().to_dict()
    return json.dumps(summary_dict, indent=4)

In [ ]:
# Group Wise Descriptive Statistics for each feature
def group_wise_stats(df, label_col):
    if label_col not in df.columns:
      raise ValueError(f"{label_col} column not found in dataframe.")

    result = {}

    for col in df.columns[:-1]:
        stats = df.groupby(label_col)[col].describe().round(2)
        result[col] = stats.to_dict()

    return json.dumps(result, indent=4)

### t-test

In [ ]:
def t_test(df, label_col):

    if label_col not in df.columns:
        raise ValueError(f"{label_col} column not found in dataframe.")

    # Select numeric features except label
    features = df.select_dtypes(include=[np.number]).columns.drop(label_col)

    # Get unique groups
    groups = df[label_col].unique()

    if len(groups) != 2:
        raise ValueError("Independent t-test requires exactly two groups.")

    g1_label, g0_label = groups

    results = {}

    for feature in features:

        # Extract actual data for each group
        group1 = df[df[label_col] == g1_label][feature].dropna()
        group0 = df[df[label_col] == g0_label][feature].dropna()

        if len(group1) < 2 or len(group0) < 2:
            continue

        t_stat, p_value = ttest_ind(group1, group0, equal_var=False)

        pooled_std = np.sqrt((group1.var() + group0.var()) / 2)

        cohens_d = 0.0 if pooled_std == 0 else \
            (group1.mean() - group0.mean()) / pooled_std

        results[feature] = {
            "group1_label": g1_label,
            "group0_label": g0_label,
            "mean_group1": float(round(group1.mean(), 3)),
            "mean_group0": float(round(group0.mean(), 3)),
            "t_statistic": float(round(t_stat, 3)),
            "p_value": float(round(p_value, 6)),
            "cohens_d": float(round(cohens_d, 3)),
            "significant": bool(p_value < 0.05)
        }

    return results

###  Mann–Whitney U Test

In [ ]:
def mann_whitney_test(df, label_col):

    if label_col not in df.columns:
        raise ValueError(f"Label column '{label_col}' not found.")

    if df[label_col].nunique() != 2:
        raise ValueError("Mann–Whitney test requires exactly 2 groups.")

    results = {}

    # Numeric columns except label
    numeric_cols = df.select_dtypes(include=[np.number]).columns.drop(label_col)

    for feature in numeric_cols:

        group_values = df[[feature, label_col]].dropna()

        groups = group_values[label_col].unique()
        g0 = group_values[group_values[label_col] == groups[0]][feature]
        g1 = group_values[group_values[label_col] == groups[1]][feature]

        if len(g0) < 1 or len(g1) < 1:
            continue

        try:
            u_stat, p_val = mannwhitneyu(g0, g1, alternative='two-sided')
        except Exception:
            continue

        results[feature] = {
            "median_group1": float(round(g0.median(), 3)),
            "median_group2": float(round(g1.median(), 3)),
            "U_statistic": float(round(u_stat, 3)),
            "p_value": float(round(p_val, 6)),
            "significant": bool(p_val < 0.05)
        }

    return results

### MANOVA Test

In [ ]:
def manova(df, label_col):

    if label_col not in df.columns:
        raise ValueError(f"{label_col} not found in dataframe.")

    if df[label_col].nunique() < 2:
        raise ValueError("MANOVA requires at least two groups.")

    # Select numeric features except label
    numeric_cols = df.select_dtypes(include=[np.number]).columns.drop(label_col)

    if len(numeric_cols) < 1:
        raise ValueError("No numeric dependent variables found.")

    # Build formula dynamically
    dependent_vars = " + ".join(numeric_cols)
    formula = f"{dependent_vars} ~ {label_col}"

    try:
        manova = MANOVA.from_formula(formula, data=df)
        result = manova.mv_test()
    except Exception as e:
        raise RuntimeError(f"MANOVA failed: {str(e)}")

    output = {
        "method": "MANOVA",
        "effects": {}
    }

    for effect, effect_data in result.results.items():
        stat_table = effect_data["stat"]

        effect_dict = {}

        for test_name, row in stat_table.iterrows():

            # Normalize test name
            normalized_name = (
                test_name.lower()
                .replace(" ", "_")
                .replace("-", "_")
                .replace("'", "")
            )

            effect_dict[normalized_name] = {
                "Value": float(row["Value"]),
                "Num DF": float(row["Num DF"]),
                "Den DF": float(row["Den DF"]),
                "F Value": float(row["F Value"]),
                "Pr > F": float(row["Pr > F"])
            }

        output["effects"][effect] = effect_dict

    return output

### PERMANOVA Test

In [ ]:
def permanova_test(df, label_col, metric="euclidean", permutations=999):

    # dataframe validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    if label_col not in df.columns:
        raise ValueError(f"Label column '{label_col}' not found.")

    # group validation
    if df[label_col].nunique() < 2:
        raise ValueError("PERMANOVA requires at least two groups.")

    # automatically select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features found for PERMANOVA.")

    # missing value check
    cols = feature_cols + [label_col]
    if df[cols].isnull().any().any():
        raise ValueError("Missing values detected. Please clean the dataset.")

    # feature matrix
    X = df[feature_cols].values
    groups = df[label_col].values

    try:
        dist_matrix = squareform(pdist(X, metric=metric))
        dm = DistanceMatrix(dist_matrix)

        result = permanova(
            distance_matrix=dm,
            grouping=groups,
            permutations=permutations
        )

    except Exception as e:
        raise RuntimeError(f"PERMANOVA failed: {str(e)}")

    return {
        "method_name": "PERMANOVA",
        "metric": metric,
        "permutations": permutations,
        "test_statistic_name": "pseudo-F",
        "test_statistic": float(result["test statistic"]),
        "p_value": float(result["p-value"]),
        "sample_size": int(result["sample size"]),
        "number_of_groups": int(result["number of groups"]),
        "significant": bool(result["p-value"] < 0.05)
    }

# Correlation Heatmap

In [ ]:
def correlation_matrix(df, label_col=None, method="pearson"):

    if method not in ["pearson","spearman","kendall"]:
        raise ValueError("Method must be 'pearson', 'spearman', or 'kendall'.")

    # dataframe validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    # select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    # remove label column if numeric
    if label_col is not None and label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) < 2:
        raise ValueError("At least two numeric features are required to compute correlation.")

    # missing value check
    if df[feature_cols].isnull().any().any():
        raise ValueError("Missing values detected in dataset.")

    # compute correlation matrix
    corr_matrix = df[feature_cols].corr(method=method).round(3)

    return {
        "method_used": method,
        "features": feature_cols,
        "correlation_matrix": corr_matrix.to_dict()
    }

In [ ]:
def statistical_modelliing(df, test_name, label_col=None, **kwargs):
    if test_name == "df_summary":
        return statistics(df)

    elif test_name == "group_wise_summary":
        return group_wise_stats(df, label_col)

    elif test_name == "t_test":
        return t_test(df, label_col)

    elif test_name == "mann_whitney_test":
        return mann_whitney_test(df, label_col)

    elif test_name == "manova":
        return manova(df, label_col)

    elif test_name == "permanova":
        return permanova_test(df, label_col, metric=kwargs.get("metric", "euclidean"), permutations=kwargs.get("permutations", 999))

    elif test_name == "correlation_matrix":
        return correlation_matrix(df, label_col, method=kwargs.get("method", "pearson"))

    else:
        raise ValueError(f"Invalid test name provided: {test_name}")

In [ ]:
statistical_modelliing(df, test_name="df_summary")

'{\n    "Permeability": {\n        "count": 14.0,\n        "mean": 45.857142857142854,\n        "std": 26.72036717520966,\n        "min": 13.0,\n        "25%": 21.25,\n        "50%": 47.0,\n        "75%": 69.75,\n        "max": 79.0\n    },\n    "LDL_uptake": {\n        "count": 14.0,\n        "mean": 0.9571428571428572,\n        "std": 0.6532101625690743,\n        "min": 0.19,\n        "25%": 0.31,\n        "50%": 1.15,\n        "75%": 1.375,\n        "max": 2.0\n    },\n    "Total_ROS": {\n        "count": 14.0,\n        "mean": 1.4499999999999997,\n        "std": 0.54877485925819,\n        "min": 0.7,\n        "25%": 0.925,\n        "50%": 1.5,\n        "75%": 1.9,\n        "max": 2.2\n    },\n    "Vascular_Marker": {\n        "count": 14.0,\n        "mean": 84.71428571428571,\n        "std": 12.418525686239096,\n        "min": 68.0,\n        "25%": 74.25,\n        "50%": 85.5,\n        "75%": 96.0,\n        "max": 100.0\n    },\n    "Cell_Signalling": {\n        "count": 14.0,\n   

In [ ]:
statistical_modelliing(df, test_name="group_wise_summary", label_col="Group")

'{\n    "Permeability": {\n        "count": {\n            "0": 7.0,\n            "1": 7.0\n        },\n        "mean": {\n            "0": 71.29,\n            "1": 20.43\n        },\n        "std": {\n            "0": 4.27,\n            "1": 4.47\n        },\n        "min": {\n            "0": 67.0,\n            "1": 13.0\n        },\n        "25%": {\n            "0": 68.5,\n            "1": 18.5\n        },\n        "50%": {\n            "0": 70.0,\n            "1": 21.0\n        },\n        "75%": {\n            "0": 73.0,\n            "1": 22.5\n        },\n        "max": {\n            "0": 79.0,\n            "1": 27.0\n        }\n    },\n    "LDL_uptake": {\n        "count": {\n            "0": 7.0,\n            "1": 7.0\n        },\n        "mean": {\n            "0": 0.53,\n            "1": 1.39\n        },\n        "std": {\n            "0": 0.65,\n            "1": 0.27\n        },\n        "min": {\n            "0": 0.19,\n            "1": 1.1\n        },\n        "25%": {\n

In [ ]:
statistical_modelliing(df, test_name="t_test", label_col="Group")

{'Permeability': {'group1_label': np.int64(0),
  'group0_label': np.int64(1),
  'mean_group1': 71.286,
  'mean_group0': 20.429,
  't_statistic': 21.773,
  'p_value': 0.0,
  'cohens_d': 11.638,
  'significant': True},
 'LDL_uptake': {'group1_label': np.int64(0),
  'group0_label': np.int64(1),
  'mean_group1': 0.529,
  'mean_group0': 1.386,
  't_statistic': -3.22,
  'p_value': 0.012307,
  'cohens_d': -1.721,
  'significant': True},
 'Total_ROS': {'group1_label': np.int64(0),
  'group0_label': np.int64(1),
  'mean_group1': 1.957,
  'mean_group0': 0.943,
  't_statistic': 11.725,
  'p_value': 0.0,
  'cohens_d': 6.267,
  'significant': True},
 'Vascular_Marker': {'group1_label': np.int64(0),
  'group0_label': np.int64(1),
  'mean_group1': 73.429,
  'mean_group0': 96.0,
  't_statistic': -9.824,
  'p_value': 1e-06,
  'cohens_d': -5.251,
  'significant': True},
 'Cell_Signalling': {'group1_label': np.int64(0),
  'group0_label': np.int64(1),
  'mean_group1': 49.429,
  'mean_group0': 86.714,
  't

In [ ]:
statistical_modelliing(df, test_name="mann_whitney_test", label_col="Group")

{'Permeability': {'median_group1': 70.0,
  'median_group2': 21.0,
  'U_statistic': 49.0,
  'p_value': 0.000583,
  'significant': True},
 'LDL_uptake': {'median_group1': 0.3,
  'median_group2': 1.3,
  'U_statistic': 7.0,
  'p_value': 0.029483,
  'significant': True},
 'Total_ROS': {'median_group1': 1.9,
  'median_group2': 0.9,
  'U_statistic': 49.0,
  'p_value': 0.002093,
  'significant': True},
 'Vascular_Marker': {'median_group1': 74.0,
  'median_group2': 96.0,
  'U_statistic': 0.0,
  'p_value': 0.002117,
  'significant': True},
 'Cell_Signalling': {'median_group1': 49.0,
  'median_group2': 88.0,
  'U_statistic': 0.0,
  'p_value': 0.002141,
  'significant': True},
 'Tube_formation': {'median_group1': 40.0,
  'median_group2': 97.0,
  'U_statistic': 0.0,
  'p_value': 0.000583,
  'significant': True},
 'In_vivo_recovery': {'median_group1': 28.0,
  'median_group2': 78.0,
  'U_statistic': 0.0,
  'p_value': 0.000583,
  'significant': True}}

In [ ]:
statistical_modelliing(df, test_name="manova", label_col="Group")

{'method': 'MANOVA',
 'effects': {'Intercept': {'wilks_lambda': {'Value': 0.0006117998357473287,
    'Num DF': 7.0,
    'Den DF': 6.0,
    'F Value': 1400.1613064136648,
    'Pr > F': 3.30233990541482e-09},
   'pillais_trace': {'Value': 0.9993882001642527,
    'Num DF': 7.0,
    'Den DF': 6.0,
    'F Value': 1400.1613064136645,
    'Pr > F': 3.3023399054148214e-09},
   'hotelling_lawley_trace': {'Value': 1633.5215241492754,
    'Num DF': 7.0,
    'Den DF': 6.0,
    'F Value': 1400.1613064136645,
    'Pr > F': 3.3023399054148214e-09},
   'roys_greatest_root': {'Value': 1633.5215241492754,
    'Num DF': 7.0,
    'Den DF': 6.0,
    'F Value': 1400.1613064136645,
    'Pr > F': 3.3023399054148214e-09}},
  'Group': {'wilks_lambda': {'Value': 0.0021848870754259675,
    'Num DF': 7.0,
    'Den DF': 6.0,
    'F Value': 391.4481926374835,
    'Pr > F': 1.4996797001249033e-07},
   'pillais_trace': {'Value': 0.997815112924574,
    'Num DF': 7.0,
    'Den DF': 6.0,
    'F Value': 391.4481926374835,

In [ ]:
statistical_modelliing(df, test_name="permanova", label_col="Group", metric="euclidean", permutations=999)

{'method_name': 'PERMANOVA',
 'metric': 'euclidean',
 'permutations': 999,
 'test_statistic_name': 'pseudo-F',
 'test_statistic': 277.62276485343426,
 'p_value': 0.001,
 'sample_size': 14,
 'number_of_groups': 2,
 'significant': True}

In [ ]:
statistical_modelliing(df, test_name="correlation_matrix", label_col="Group", method="pearson")

{'method_used': 'pearson',
 'features': ['Permeability',
  'LDL_uptake',
  'Total_ROS',
  'Vascular_Marker',
  'Cell_Signalling',
  'Tube_formation',
  'In_vivo_recovery'],
 'correlation_matrix': {'Permeability': {'Permeability': 1.0,
   'LDL_uptake': -0.712,
   'Total_ROS': 0.959,
   'Vascular_Marker': -0.95,
   'Cell_Signalling': -0.962,
   'Tube_formation': -0.982,
   'In_vivo_recovery': -0.948},
  'LDL_uptake': {'Permeability': -0.712,
   'LDL_uptake': 1.0,
   'Total_ROS': -0.598,
   'Vascular_Marker': 0.642,
   'Cell_Signalling': 0.762,
   'Tube_formation': 0.708,
   'In_vivo_recovery': 0.597},
  'Total_ROS': {'Permeability': 0.959,
   'LDL_uptake': -0.598,
   'Total_ROS': 1.0,
   'Vascular_Marker': -0.914,
   'Cell_Signalling': -0.9,
   'Tube_formation': -0.946,
   'In_vivo_recovery': -0.946},
  'Vascular_Marker': {'Permeability': -0.95,
   'LDL_uptake': 0.642,
   'Total_ROS': -0.914,
   'Vascular_Marker': 1.0,
   'Cell_Signalling': 0.886,
   'Tube_formation': 0.927,
   'In_vivo_

# Anomaly Detection

## 1). Z-Score Method

In [ ]:
def zscore_outliers(df, label_col=None, threshold=3):

    # dataframe validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    # select numeric features automatically
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col is not None and label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features available for Z-score analysis.")

    # check missing values
    if df[feature_cols].isnull().any().any():
        raise ValueError("Missing values detected in numeric features.")

    # feature matrix
    X = df[feature_cols]

    # compute Z-scores
    z_scores = np.abs(stats.zscore(X))
    anomaly_mask = (z_scores > threshold).any(axis=1)

    anomalies = df.loc[anomaly_mask].copy()

    return {
        "threshold": threshold,
        "total_rows": int(len(df)),
        "num_anomalies": int(anomaly_mask.sum()),
        "anomaly_rows": anomalies.to_dict(orient="records")
    }

## 2). IQR Outlier Detection (Box Plot)

In [ ]:
def boxplot(df, label_col):

    # dataframe validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    if label_col not in df.columns:
        raise ValueError(f"{label_col} not found in dataframe.")

    # automatically select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features available for boxplot.")

    # check missing values
    if df[feature_cols].isnull().any().any():
        raise ValueError("Missing values detected in numeric features.")

    results = {}

    for feature in feature_cols:
        results[feature] = {}

        for group, group_df in df.groupby(label_col):

            values = group_df[feature]

            q1 = np.percentile(values, 25)
            median = np.percentile(values, 50)
            q3 = np.percentile(values, 75)
            iqr = q3 - q1

            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr

            # TRUE whiskers (closest value within bounds)
            lower_whisker = values[values >= lower_bound].min()
            upper_whisker = values[values <= upper_bound].max()

            # Outliers
            outlier_mask = (values < lower_bound) | (values > upper_bound)
            outlier_rows = group_df.loc[outlier_mask]

            results[feature][str(group)] = {
                "min": float(values.min()),
                "q1": float(q1),
                "median": float(median),
                "q3": float(q3),
                "max": float(values.max()),
                "lower_whisker": float(lower_whisker),
                "upper_whisker": float(upper_whisker),
                "outliers": values[outlier_mask].tolist(),
                "outlier_rows": outlier_rows.to_dict(orient="records"),
                "count": int(len(values))
            }

    return results

## 3). Isolation Forest

In [ ]:
def isolation_forest(df, label_col=None, contamination=0.1, random_state=42):

    # dataframe validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    # automatically select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col is not None and label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric features available for Isolation Forest.")

    # missing value check
    if df[feature_cols].isnull().any().any():
        raise ValueError("Missing values detected in numeric features.")

    # feature matrix
    X = df[feature_cols]

    # standardization (your existing function)
    X_scaled, scaler = standardization(X)

    # Isolation Forest model
    model = IsolationForest(
        contamination=contamination,
        random_state=random_state
    )

    model.fit(X_scaled)

    predictions = model.predict(X_scaled)

    anomaly_mask = predictions == -1
    anomalies = df.loc[anomaly_mask].copy()

    result = {
        "contamination": contamination,
        "total_rows": int(len(df)),
        "num_anomalies": int(anomaly_mask.sum()),
        "anomaly_indices": anomalies.index.tolist(),
        "anomaly_rows": anomalies.to_dict(orient="records"),
    }

    # group-wise anomaly count
    if label_col is not None and label_col in df.columns:

        groupwise_counts = {}

        for group, group_df in df.groupby(label_col):

            group_mask = anomaly_mask[df[label_col] == group]

            groupwise_counts[str(group)] = {
                "group_size": int(len(group_df)),
                "num_anomalies": int(group_mask.sum())
            }

        result["groupwise_anomalies"] = groupwise_counts

    return result

In [ ]:
def ananomaly_detection(df, test_name, label_col=None, **kwargs):
    if test_name == "z_score":
        return zscore_outliers(df, label_col, threshold=kwargs.get("threshold", 3))

    elif test_name == "boxplot":
        return boxplot(df, label_col)

    elif test_name == "isolation_forest":
        return isolation_forest(df, label_col, contamination=kwargs.get("contamination", 0.1), random_state=kwargs.get("random_state", 42))

    else:
        raise ValueError(f"Invalid test name: {test_name}")

In [ ]:
ananomaly_detection(df, test_name="z_score", label_col="Group", threshold=3)

{'threshold': 3, 'total_rows': 14, 'num_anomalies': 0, 'anomaly_rows': []}

In [ ]:
ananomaly_detection(df, test_name="boxplot", label_col="Group")

{'Permeability': {'0': {'min': 67.0,
   'q1': 68.5,
   'median': 70.0,
   'q3': 73.0,
   'max': 79.0,
   'lower_whisker': 67.0,
   'upper_whisker': 79.0,
   'outliers': [],
   'outlier_rows': [],
   'count': 7},
  '1': {'min': 13.0,
   'q1': 18.5,
   'median': 21.0,
   'q3': 22.5,
   'max': 27.0,
   'lower_whisker': 13.0,
   'upper_whisker': 27.0,
   'outliers': [],
   'outlier_rows': [],
   'count': 7}},
 'LDL_uptake': {'0': {'min': 0.19,
   'q1': 0.26,
   'median': 0.3,
   'q3': 0.345,
   'max': 2.0,
   'lower_whisker': 0.19,
   'upper_whisker': 0.35,
   'outliers': [2.0],
   'outlier_rows': [{'Permeability': 69,
     'LDL_uptake': 2.0,
     'Total_ROS': 2.2,
     'Vascular_Marker': 74,
     'Cell_Signalling': 55,
     'Tube_formation': 44,
     'In_vivo_recovery': 19,
     'Group': 0}],
   'count': 7},
  '1': {'min': 1.1,
   'q1': 1.2,
   'median': 1.3,
   'q3': 1.5499999999999998,
   'max': 1.8,
   'lower_whisker': 1.1,
   'upper_whisker': 1.8,
   'outliers': [],
   'outlier_rows':

In [ ]:
ananomaly_detection(df, test_name="isolation_forest", label_col="Group", contamination=0.05, random_state=42)

{'contamination': 0.05,
 'total_rows': 14,
 'num_anomalies': 1,
 'anomaly_indices': [5],
 'anomaly_rows': [{'Permeability': 69,
   'LDL_uptake': 2.0,
   'Total_ROS': 2.2,
   'Vascular_Marker': 74,
   'Cell_Signalling': 55,
   'Tube_formation': 44,
   'In_vivo_recovery': 19,
   'Group': 0}],
 'groupwise_anomalies': {'0': {'group_size': 7, 'num_anomalies': 1},
  '1': {'group_size': 7, 'num_anomalies': 0}}}

# Predictive Modelling

## Classification

In [ ]:
# Validation
def validate_input(df, label_col):
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")
    if df.empty:
        raise ValueError("Dataset is empty.")
    if label_col not in df.columns:
        raise ValueError(f"{label_col} not found in dataset.")
    if df[label_col].nunique() != 2:
        raise ValueError("Binary classification requires exactly 2 classes.")
    if len(df) < 10:
        raise ValueError("Dataset too small for modeling.")

Multicollinearity

In [ ]:
def compute_vif(df, label_col):
    if label_col not in df.columns:
        raise ValueError(f"{label_col} not found in dataset.")

    X = df.drop(columns=[label_col])
    X = X.select_dtypes(include=[np.number])

    #if X.isnull().any().any():
    #    X = X.fillna(X.mean())

    vif_values = []

    for i in range(X.shape[1]):
        vif_score = float(variance_inflation_factor(X.values, i))
        vif_values.append({
            "feature": X.columns[i],
            "value": round(vif_score, 2)
        })

    # Sort descending
    vif_values = sorted(vif_values, key=lambda x: x["value"], reverse=True)

    return {
        "vif": vif_values
    }

In [ ]:
compute_vif(df, label_col="Group")

{'vif': [{'feature': 'Vascular_Marker', 'value': 216.07},
  {'feature': 'Cell_Signalling', 'value': 190.91},
  {'feature': 'Tube_formation', 'value': 179.21},
  {'feature': 'Total_ROS', 'value': 155.26},
  {'feature': 'Permeability', 'value': 106.44},
  {'feature': 'In_vivo_recovery', 'value': 80.37},
  {'feature': 'LDL_uptake', 'value': 10.3}]}

1). Logistic Regression

a) Ridge

In [ ]:
def ridge_logistic_classification(df, label_col, test_data_size=0.28):

    # Feature Selection
    feature_cols = df.select_dtypes(include=[np.number]).columns.drop(label_col)
    X = df[feature_cols].copy()
    y = df[label_col]

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_data_size, random_state=42, stratify=y)

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Adaptive CV Folds
    n_samples = len(X_train)

    if n_samples < 50:
        num_splits = 3
    else:
        num_splits = 5

    skf = StratifiedKFold(n_splits=num_splits, shuffle=True, random_state=42)

    # Grid Search
    total_samples = len(df)
    if total_samples < 50:
        # Conservative: prevents extreme over/underfitting
        c_range = np.logspace(-1, 1, 10)
    elif total_samples < 500:
        c_range = np.logspace(-2, 2, 15)
    else:
        # Wide search for big data
        c_range = np.logspace(-3, 3, 20)

    parameter_grid = {"C": c_range}

    base_model = LogisticRegression(penalty="l2", solver="lbfgs", C=1, max_iter=1000, random_state=42)

    grid = GridSearchCV(estimator=base_model, param_grid=parameter_grid, cv=skf, scoring="roc_auc", n_jobs=-1, verbose=1, return_train_score=True)

    grid.fit(X_train_scaled, y_train)

    # Extract CV Curve Data
    cv_results = pd.DataFrame(grid.cv_results_)

    c_values = cv_results["param_C"].astype(float)
    train_scores = cv_results["mean_train_score"]
    val_scores = cv_results["mean_test_score"]

    # Sort by C for frontend plotting
    sorted_idx = np.argsort(c_values)
    c_values = c_values.iloc[sorted_idx]
    train_scores = train_scores.iloc[sorted_idx]
    val_scores = val_scores.iloc[sorted_idx]

    best_model = grid.best_estimator_
    best_C = grid.best_params_["C"]
    best_cv_score = grid.best_score_

    # Final Test Evaluation
    y_pred = best_model.predict(X_test_scaled)
    y_prob = best_model.predict_proba(X_test_scaled)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, output_dict=True)

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    # Coefficients
    coef_series = pd.Series(best_model.coef_[0], index=feature_cols)
    coef_sorted = coef_series.sort_values(ascending=False)

    # Final JSON Output
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y.value_counts().to_dict()
        },
        "model_selection": {
            "cv_folds": num_splits,
            "best_C": float(best_C),
            "best_cv_auc": float(best_cv_score),
            "c_selection_curve": {
                "C_values": c_values.tolist(),
                "train_auc": train_scores.tolist(),
                "validation_auc": val_scores.tolist()
            }
        },
        "model_results": {
            "model_type": "Logistic Regression (Ridge)",
            "accuracy": float(accuracy),
            "auc_score": float(auc_score),
            "confusion_matrix": conf_matrix.tolist(),
            "classification_report": class_report,
            "roc_curve": {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist()
            },
            "coefficients": coef_sorted.to_dict()
        }
    }

b) Lasso

In [ ]:
def lasso_logistic_classification(df, label_col, test_data_size=0.28):

    # Feature Selection
    feature_cols = df.select_dtypes(include=[np.number]).columns.drop(label_col)
    X = df[feature_cols].copy()
    y = df[label_col]

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_data_size, random_state=42, stratify=y)

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Adaptive CV Folds
    n_samples = len(X_train)

    if n_samples < 50:
        num_splits = 3
    else:
        num_splits = 5

    skf = StratifiedKFold(n_splits=num_splits, shuffle=True, random_state=42)

    # Grid Search
    total_samples = len(df)
    if total_samples < 50:
        # Conservative: prevents extreme over/underfitting
        c_range = np.logspace(-1, 1, 10)
    elif total_samples < 500:
        c_range = np.logspace(-2, 2, 15)
    else:
        # Wide search for big data
        c_range = np.logspace(-3, 3, 20)

    parameter_grid = {"C": c_range}

    base_model = LogisticRegression(penalty="l1", solver="liblinear", C=1, max_iter=1000, random_state=42)

    grid = GridSearchCV(estimator=base_model, param_grid=parameter_grid, cv=skf, scoring="roc_auc", n_jobs=-1, verbose=1, return_train_score=True)

    grid.fit(X_train_scaled, y_train)

    # Extract CV Curve Data
    cv_results = pd.DataFrame(grid.cv_results_)

    c_values = cv_results["param_C"].astype(float)
    train_scores = cv_results["mean_train_score"]
    val_scores = cv_results["mean_test_score"]

    # Sort by C for frontend plotting
    sorted_idx = np.argsort(c_values)
    c_values = c_values.iloc[sorted_idx]
    train_scores = train_scores.iloc[sorted_idx]
    val_scores = val_scores.iloc[sorted_idx]

    best_model = grid.best_estimator_
    best_C = grid.best_params_["C"]
    best_cv_score = grid.best_score_

    # Final Test Evaluation
    y_pred = best_model.predict(X_test_scaled)
    y_prob = best_model.predict_proba(X_test_scaled)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, output_dict=True)

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    # Coefficients
    coef_series = pd.Series(best_model.coef_[0], index=feature_cols)
    coef_sorted = coef_series.sort_values(ascending=False)

    # Final JSON Output
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y.value_counts().to_dict()
        },
        "model_selection": {
            "cv_folds": num_splits,
            "best_C": float(best_C),
            "best_cv_auc": float(best_cv_score),
            "c_selection_curve": {
                "C_values": c_values.tolist(),
                "train_auc": train_scores.tolist(),
                "validation_auc": val_scores.tolist()
            }
        },
        "model_results": {
            "model_type": "Logistic Regression (Ridge)",
            "accuracy": float(accuracy),
            "auc_score": float(auc_score),
            "confusion_matrix": conf_matrix.tolist(),
            "classification_report": class_report,
            "roc_curve": {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist()
            },
            "coefficients": coef_sorted.to_dict()
        }
    }

c) ElasticNet

In [ ]:
def elasticnet_logistic_classification(df, label_col, test_data_size=0.28):

    # Feature Selection
    feature_cols = df.select_dtypes(include=[np.number]).columns.drop(label_col)
    X = df[feature_cols].copy()
    y = df[label_col]

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_data_size, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    if len(X_train) < 50:
        num_splits = 3
    else:
        num_splits = 5

    skf = StratifiedKFold(n_splits=num_splits, shuffle=True, random_state=42)

    # Grid Search
    total_samples = len(df)

    if total_samples < 50:
        c_range = np.logspace(-1, 1, 10)
    elif total_samples < 500:
        c_range = np.logspace(-2, 2, 15)
    else:
        c_range = np.logspace(-3, 3, 20)

    # Elastic Net requires l1_ratio
    l1_ratios = [0.2, 0.5, 0.8]

    parameter_grid = {
        "C": c_range,
        "l1_ratio": l1_ratios
    }

    # Base Model
    base_model = LogisticRegression(penalty="elasticnet", solver="saga", max_iter=2000, C=1, random_state=42)

    grid = GridSearchCV(estimator=base_model, param_grid=parameter_grid, cv=skf, scoring="roc_auc", n_jobs=-1, verbose=1, return_train_score=True)

    grid.fit(X_train_scaled, y_train)

    # Extract CV Curve Data
    cv_results = pd.DataFrame(grid.cv_results_)

    # For frontend, show curve for best l1_ratio
    best_l1_ratio = grid.best_params_["l1_ratio"]

    filtered_results = cv_results[cv_results["param_l1_ratio"] == best_l1_ratio]

    c_values = filtered_results["param_C"].astype(float)
    train_scores = filtered_results["mean_train_score"]
    val_scores = filtered_results["mean_test_score"]

    sorted_idx = np.argsort(c_values)
    c_values = c_values.iloc[sorted_idx]
    train_scores = train_scores.iloc[sorted_idx]
    val_scores = val_scores.iloc[sorted_idx]

    best_model = grid.best_estimator_
    best_C = grid.best_params_["C"]
    best_cv_score = grid.best_score_

    # Final Test Evaluation
    y_pred = best_model.predict(X_test_scaled)
    y_prob = best_model.predict_proba(X_test_scaled)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, output_dict=True)

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    # Coefficients
    coef_series = pd.Series(best_model.coef_[0], index=feature_cols)
    coef_sorted = coef_series.sort_values(ascending=False)

    # Final JSON Output
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y.value_counts().to_dict()
        },
        "model_selection": {
            "cv_folds": num_splits,
            "best_C": float(best_C),
            "best_l1_ratio": float(best_l1_ratio),
            "best_cv_auc": float(best_cv_score),
            "c_selection_curve": {
                "C_values": c_values.tolist(),
                "train_auc": train_scores.tolist(),
                "validation_auc": val_scores.tolist()
            }
        },
        "model_results": {
            "model_type": "Logistic Regression (Elastic Net)",
            "accuracy": float(accuracy),
            "auc_score": float(auc_score),
            "confusion_matrix": conf_matrix.tolist(),
            "classification_report": class_report,
            "roc_curve": {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist()
            },
            "coefficients": coef_sorted.to_dict()
        }
    }

In [ ]:
def logistic_classification(model, df, label_col="Group", test_data_size=0.28, **kwargs):
    """
    Central classification dispatcher.

    Parameters:
    - model: string selected by user
    - df: full dataframe
    - label_col: target column
    - test_data_size: test split ratio
    - kwargs: algorithm specific parameters
    """

    # Validate Input
    validate_input(df, label_col)

    # Check Multicollinearity
    compute_vif(df, label_col)

    # Model Selection
    if model == "Ridge Logistic Regression":
        return ridge_logistic_classification(df=df, label_col=label_col, test_data_size=test_data_size)

    elif model == "Lasso Logistic Regression":
        return lasso_logistic_classification(df=df, label_col=label_col, test_data_size=test_data_size)

    elif model == "ElasticNet Logistic Regression":
        return elasticnet_logistic_classification(df=df, label_col=label_col, test_data_size=test_data_size)
    else:
        raise ValueError(
            "Invalid model. Choose from: "
            "'Ridge Logistic Regression', "
            "'Lasso Logistic Regression', "
            "'ElasticNet Logistic Regression'"
        )

In [ ]:
model = input("Enter the model: ")
logistic_classification(model=model, df=df, label_col="Group", test_data_size=0.28)

Enter the model: Ridge Logistic Regression
Fitting 3 folds for each of 10 candidates, totalling 30 fits


{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'model_selection': {'cv_folds': 3,
  'best_C': 0.1,
  'best_cv_auc': 1.0,
  'c_selection_curve': {'C_values': [0.1,
    0.16681005372000587,
    0.2782559402207124,
    0.46415888336127786,
    0.774263682681127,
    1.291549665014884,
    2.1544346900318834,
    3.593813663804626,
    5.994842503189409,
    10.0],
   'train_auc': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
   'validation_auc': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}},
 'model_results': {'model_type': 'Logistic Regression (Ridge)',
  'accuracy': 1.0,
  'auc_score': 1.0,
  'confusion_matrix': [[2, 0], [0, 2]],
  'classification_report': {'0': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 2.0},
   '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 2.0},
   'accuracy': 1.0,
   'macro avg': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 

In [ ]:
model = input("Enter the model: ")
logistic_classification(model=model, df=df, label_col="Group", test_data_size=0.28)

Enter the model: Lasso Logistic Regression
Fitting 3 folds for each of 10 candidates, totalling 30 fits


{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'model_selection': {'cv_folds': 3,
  'best_C': 0.46415888336127786,
  'best_cv_auc': 1.0,
  'c_selection_curve': {'C_values': [0.1,
    0.16681005372000587,
    0.2782559402207124,
    0.46415888336127786,
    0.774263682681127,
    1.291549665014884,
    2.1544346900318834,
    3.593813663804626,
    5.994842503189409,
    10.0],
   'train_auc': [0.5,
    0.5,
    0.8333333333333334,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0],
   'validation_auc': [0.5,
    0.5,
    0.8333333333333334,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0]}},
 'model_results': {'model_type': 'Logistic Regression (Ridge)',
  'accuracy': 1.0,
  'auc_score': 1.0,
  'confusion_matrix': [[2, 0], [0, 2]],
  'classification_report': {'0': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 2.0},
   '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support'

In [ ]:
model = input("Enter the model: ")
logistic_classification(model=model, df=df, label_col="Group", test_data_size=0.28)

Enter the model: ElasticNet Logistic Regression
Fitting 3 folds for each of 30 candidates, totalling 90 fits


{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'model_selection': {'cv_folds': 3,
  'best_C': 0.1,
  'best_l1_ratio': 0.2,
  'best_cv_auc': 1.0,
  'c_selection_curve': {'C_values': [0.1,
    0.16681005372000587,
    0.2782559402207124,
    0.46415888336127786,
    0.774263682681127,
    1.291549665014884,
    2.1544346900318834,
    3.593813663804626,
    5.994842503189409,
    10.0],
   'train_auc': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
   'validation_auc': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}},
 'model_results': {'model_type': 'Logistic Regression (Elastic Net)',
  'accuracy': 1.0,
  'auc_score': 1.0,
  'confusion_matrix': [[2, 0], [0, 2]],
  'classification_report': {'0': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 2.0},
   '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 2.0},
   'accuracy': 1.0,
   'macro avg': {'precision': 1.0,
    'recall': 1.0,
    'f

2). SVM

In [ ]:
def svm_classification(df, label_col, test_data_size=0.28, user_params=None):
    validate_input(df, label_col)

    # Feature Selection
    feature_cols = df.select_dtypes(include=[np.number]).columns.drop(label_col)
    X = df[feature_cols].copy()
    y = df[label_col]

    # Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_data_size, random_state=42, stratify=y)

    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # If User Provides Parameters
    if user_params is not None:
        model = SVC(
            kernel=user_params.get("kernel", "rbf"),
            C=user_params.get("C", 1.0),
            gamma=user_params.get("gamma", "scale"),
            probability=True,
            random_state=42
        )

        model.fit(X_train_scaled, y_train)

        best_model = model
        best_params = {
            "kernel": user_params.get("kernel", "rbf"),
            "C": user_params.get("C", 1.0),
            "gamma": user_params.get("gamma", "scale")
        }

        cv_curve_data = None
        best_cv_score = None
        num_splits = None

    # Automatic Mode (Recommended)
    else:
        # Adaptive CV folds
        if len(X_train) < 50:
            num_splits = 3
        else:
            num_splits = 5

        skf = StratifiedKFold(n_splits=num_splits, shuffle=True, random_state=42)

        # Adaptive C range
        total_samples = len(df)

        if total_samples < 50:
            c_range = np.logspace(-1, 1, 10)
        elif total_samples < 500:
            c_range = np.logspace(-2, 2, 15)
        else:
            c_range = np.logspace(-3, 3, 20)

        parameter_grid = [
            {
                "kernel": ["linear"],
                "C": c_range
            },
            {
                "kernel": ["rbf"],
                "C": c_range,
                "gamma": ["scale", 0.01, 0.1, 1]
            }
        ]

        base_model = SVC(probability=True, random_state=42)

        grid = GridSearchCV(estimator=base_model, param_grid=parameter_grid, cv=skf, scoring="roc_auc", n_jobs=-1, return_train_score=True, verbose=1)

        grid.fit(X_train_scaled, y_train)

        best_model = grid.best_estimator_
        best_params = grid.best_params_
        best_cv_score = grid.best_score_

        # Extract CV curve (for best kernel only)
        cv_results = pd.DataFrame(grid.cv_results_)
        best_kernel = best_params["kernel"]

        filtered = cv_results[cv_results["param_kernel"] == best_kernel]

        c_vals = filtered["param_C"].astype(float)
        train_scores = filtered["mean_train_score"]
        val_scores = filtered["mean_test_score"]

        sorted_idx = np.argsort(c_vals)

        cv_curve_data = {
            "C_values": c_vals.iloc[sorted_idx].tolist(),
            "train_auc": train_scores.iloc[sorted_idx].tolist(),
            "validation_auc": val_scores.iloc[sorted_idx].tolist()
        }

    # Final Test Evaluation
    y_pred = best_model.predict(X_test_scaled)
    y_prob = best_model.predict_proba(X_test_scaled)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    conf_matrix = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred, output_dict=True)

    fpr, tpr, _ = roc_curve(y_test, y_prob)

    # Final JSON Output
    return {
        "data_summary": {
            "num_samples": int(len(df)),
            "num_features": int(len(feature_cols)),
            "class_distribution": y.value_counts().to_dict()
        },
        "model_selection": {
            "cv_folds": num_splits,
            "best_kernel": best_params.get("kernel"),
            "best_C": float(best_params.get("C")),
            "best_gamma": best_params.get("gamma"),
            "best_cv_auc": float(best_cv_score) if best_cv_score else None,
            "c_selection_curve": cv_curve_data
        },
        "model_results": {
            "model_type": "SVM",
            "accuracy": float(accuracy),
            "auc_score": float(auc_score),
            "confusion_matrix": conf_matrix.tolist(),
            "classification_report": class_report,
            "roc_curve": {
                "fpr": fpr.tolist(),
                "tpr": tpr.tolist()
            }
        }
    }

In [ ]:
svm_classification(df, label_col="Group", test_data_size=0.28, user_params=None) # {"kernel": "linear", "gamma": 0.1}

Fitting 3 folds for each of 50 candidates, totalling 150 fits


{'data_summary': {'num_samples': 14,
  'num_features': 7,
  'class_distribution': {0: 7, 1: 7}},
 'model_selection': {'cv_folds': 3,
  'best_kernel': 'linear',
  'best_C': 0.1,
  'best_gamma': None,
  'best_cv_auc': 1.0,
  'c_selection_curve': {'C_values': [0.1,
    0.16681005372000587,
    0.2782559402207124,
    0.46415888336127786,
    0.774263682681127,
    1.291549665014884,
    2.1544346900318834,
    3.593813663804626,
    5.994842503189409,
    10.0],
   'train_auc': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
   'validation_auc': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}},
 'model_results': {'model_type': 'SVM',
  'accuracy': 1.0,
  'auc_score': 1.0,
  'confusion_matrix': [[2, 0], [0, 2]],
  'classification_report': {'0': {'precision': 1.0,
    'recall': 1.0,
    'f1-score': 1.0,
    'support': 2.0},
   '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 2.0},
   'accuracy': 1.0,
   'macro avg': {'precision': 1.0,
    'recall': 1.0,
    'f1-sco

3). PCA Analysis

In [ ]:
def pca_analysis_json(df, label_col, variance_threshold=0.95):
    # Select Numeric Features
    feature_cols = df.select_dtypes(include=[np.number]).columns.drop(label_col)
    X = df[feature_cols]

    # Standardize Data
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Apply PCA
    pca = PCA()
    pca.fit(X_scaled)

    explained_variance = pca.explained_variance_ratio_
    cumulative_variance = np.cumsum(explained_variance)

    # Convert to percentage
    explained_variance_pct = explained_variance * 100
    cumulative_variance_pct = cumulative_variance * 100
    threshold_pct = variance_threshold * 100

    # Compute Threshold Components
    components_needed = int(np.argmax(cumulative_variance >= variance_threshold) + 1)

    # PCA Loadings
    loadings_df = pd.DataFrame(
        pca.components_.T,
        columns=[f"PC{i+1}" for i in range(len(feature_cols))],
        index=feature_cols
    )

    # Final JSON Output
    return {
        "pca_summary": {
            "total_samples": int(len(df)),
            "total_features": int(len(feature_cols)),
            "variance_threshold_percent": float(threshold_pct),
            "components_needed_for_threshold": components_needed
        },
        "scree_plot_data": {
            "components": list(range(1, len(explained_variance) + 1)),
            "explained_variance_percent": explained_variance_pct.tolist(),
            "cumulative_variance_percent": cumulative_variance_pct.tolist(),
            "threshold_line_percent": float(threshold_pct)
        },
        "pca_loadings": loadings_df.to_dict()
    }

In [ ]:
pca_analysis_json(df, label_col="Group", variance_threshold=0.95)

{'pca_summary': {'total_samples': 14,
  'total_features': 7,
  'variance_threshold_percent': 95.0,
  'components_needed_for_threshold': 2},
 'scree_plot_data': {'components': [1, 2, 3, 4, 5, 6, 7],
  'explained_variance_percent': [88.55949128638862,
   7.553270394983713,
   1.6625140169716528,
   0.9331628222968442,
   0.6928077714880654,
   0.4628311334017339,
   0.13592257446936923],
  'cumulative_variance_percent': [88.55949128638862,
   96.11276168137233,
   97.77527569834399,
   98.70843852064083,
   99.4012462921289,
   99.86407742553062,
   100.0],
  'threshold_line_percent': 95.0},
 'pca_loadings': {'PC1': {'Permeability': 0.39899903565752076,
   'LDL_uptake': -0.30119667678031936,
   'Total_ROS': 0.3848461690720713,
   'Vascular_Marker': -0.3812643884023061,
   'Cell_Signalling': -0.38996626050911803,
   'Tube_formation': -0.39600035243844306,
   'In_vivo_recovery': -0.3843452357722018},
  'PC2': {'Permeability': 0.06412796620344198,
   'LDL_uptake': 0.9037985307994665,
   'To

In [ ]:
def pca_classification_model(df, label_col, model_type="svm", n_components=0.95):

    # Select Numeric Features
    feature_cols = df.select_dtypes(include=[np.number]).columns.drop(label_col)
    X = df[feature_cols]
    y = df[label_col]

    # Scale Data (Required for PCA)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Apply PCA
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)

    # Create PCA DataFrame
    pca_columns = [f"PC{i+1}" for i in range(X_pca.shape[1])]
    df_pca = pd.DataFrame(X_pca, columns=pca_columns)
    df_pca[label_col] = y.values

    # Choose Model Based on User Input
    model_type = model_type

    if model_type == "svm":
        model_results = svm_classification(df, label_col="Group", test_data_size=0.28, user_params={"kernel": "linear", "gamma": 0.1})

    elif model_type == "logistic":
        model = input("Enter the model: ")
        model_results = logistic_classification(model=model, df=df, label_col="Group", test_data_size=0.28)

    else:
        raise ValueError("model_type must be either 'svm' or 'logistic'")

    # Return Final JSON
    return {
        "pca_info": {
            "n_components_used": int(X_pca.shape[1]),
            "explained_variance_percent": (pca.explained_variance_ratio_ * 100).tolist(),
            "cumulative_variance_percent": (np.cumsum(pca.explained_variance_ratio_) * 100).tolist()
        },
        "selected_model": model_type,
        "model_results": model_results
    }

In [ ]:
model = input("Enter the model: ")
pca_classification_model(df, label_col="Group", model_type=model, n_components=0.95)

Enter the model: svm


{'pca_info': {'n_components_used': 2,
  'explained_variance_percent': [88.55949128638862, 7.553270394983713],
  'cumulative_variance_percent': [88.55949128638862, 96.11276168137233]},
 'selected_model': 'svm',
 'model_results': {'data_summary': {'num_samples': 14,
   'num_features': 7,
   'class_distribution': {0: 7, 1: 7}},
  'model_selection': {'cv_folds': None,
   'best_kernel': 'linear',
   'best_C': 1.0,
   'best_gamma': 0.1,
   'best_cv_auc': None,
   'c_selection_curve': None},
  'model_results': {'model_type': 'SVM',
   'accuracy': 1.0,
   'auc_score': 1.0,
   'confusion_matrix': [[2, 0], [0, 2]],
   'classification_report': {'0': {'precision': 1.0,
     'recall': 1.0,
     'f1-score': 1.0,
     'support': 2.0},
    '1': {'precision': 1.0, 'recall': 1.0, 'f1-score': 1.0, 'support': 2.0},
    'accuracy': 1.0,
    'macro avg': {'precision': 1.0,
     'recall': 1.0,
     'f1-score': 1.0,
     'support': 4.0},
    'weighted avg': {'precision': 1.0,
     'recall': 1.0,
     'f1-scor

In [ ]:
model = input("Enter the model: ")
pca_classification_model(df, label_col="Group", model_type=model, n_components=0.95)

Enter the model: logistic
Enter the model: ElasticNet Logistic Regression
Fitting 3 folds for each of 30 candidates, totalling 90 fits


{'pca_info': {'n_components_used': 2,
  'explained_variance_percent': [88.55949128638862, 7.553270394983713],
  'cumulative_variance_percent': [88.55949128638862, 96.11276168137233]},
 'selected_model': 'logistic',
 'model_results': {'data_summary': {'num_samples': 14,
   'num_features': 7,
   'class_distribution': {0: 7, 1: 7}},
  'model_selection': {'cv_folds': 3,
   'best_C': 0.1,
   'best_l1_ratio': 0.2,
   'best_cv_auc': 1.0,
   'c_selection_curve': {'C_values': [0.1,
     0.16681005372000587,
     0.2782559402207124,
     0.46415888336127786,
     0.774263682681127,
     1.291549665014884,
     2.1544346900318834,
     3.593813663804626,
     5.994842503189409,
     10.0],
    'train_auc': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
    'validation_auc': [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]}},
  'model_results': {'model_type': 'Logistic Regression (Elastic Net)',
   'accuracy': 1.0,
   'auc_score': 1.0,
   'confusion_matrix': [[2, 0], [0, 2]],
   'classifica

In [ ]:
def pca_module(df, label_col, task="analysis", **kwargs):

    # basic validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    if label_col not in df.columns:
        raise ValueError(f"{label_col} not found in dataframe.")

    # PCA analysis only
    if task == "analysis":
        return pca_analysis_json(df, label_col, variance_threshold=kwargs.get("variance_threshold", 0.95))

    # PCA-based classification (analysis + model)
    elif task == "classification":

        pca_analysis_result = pca_analysis_json(df, label_col, variance_threshold=kwargs.get("variance_threshold", 0.95))

        pca_model_result = pca_classification_model(df, label_col, model_type=kwargs.get("model_type", "svm"), n_components=kwargs.get("n_components", 0.95))

        return {
            "pca_analysis": pca_analysis_result,
            "pca_classification": pca_model_result
        }

    else:
        raise ValueError("task must be 'analysis' or 'classification'")

In [ ]:
def classification_model(df, model_name, label_col="Group", **kwargs):

    # SVM
    if model_name == "svm":
        return svm_classification(
            df,
            label_col=label_col,
            test_data_size=kwargs.get("test_data_size", 0.28),
            user_params=kwargs.get("user_params")
        )

    # Logistic Regression Family
    elif model_name == "logistic":
        return logistic_classification(
            model=kwargs.get("logistic_type"),
            df=df,
            label_col=label_col,
            test_data_size=kwargs.get("test_data_size", 0.28)
        )

    # PCA Based Models
    elif model_name == "pca_model":
        return pca_module(
            df,
            label_col=label_col,
            task="classification",
            model_type=kwargs.get("model_type", "svm"),
            n_components=kwargs.get("n_components", 0.95)
        )

    else:
        raise ValueError(
            "Invalid model_name. Choose from: "
            "'svm', 'logistic', 'pca_model'"
        )

# Clustering Analysis

# 1). K-Means Clustering

In [ ]:
# WCSS calculation for elbow plot
def elbow_info(x_scaled, max_clusters=10):
    """
    Compute WCSS and silhouette score for k in a safe range.

    - k starts from 2 (k=1 is not meaningful for clustering)
    """
    ks, wcss, silhouettes = [], [], []

    n_samples = x_scaled.shape[0]
    max_k = min(max_clusters, n_samples // 2)

    # computing the WCSS for each K
    for k in range(2, max_k + 1):
        kmeans = KMeans(n_clusters=k, init="k-means++", n_init=20, random_state=42)

        labels = kmeans.fit_predict(x_scaled)

        ks.append(k)
        wcss.append(kmeans.inertia_)
        silhouettes.append(silhouette_score(x_scaled, labels))

    # Default k chosen objectively (best silhouette)
    default_k = ks[np.argmax(silhouettes)]

    return {
        "k": ks,
        "wcss": wcss,
        "silhouette": silhouettes,
        "default_k": default_k
    }

In [ ]:
def fitting_K_means(x_scaled, k, y=None):
    # fitting the K-means model
    kmeans_model = KMeans(n_clusters=k, init="k-means++", n_init=50, random_state=42)
    labels = kmeans_model.fit_predict(x_scaled)

    # Core clustering quality metric
    metrics = {
        "silhouette": silhouette_score(x_scaled, labels)
    }

    # Supervised evaluation (if ground truth exists)
    if y is not None:
       # Adjusted Rand Index
        metrics["ari"] = adjusted_rand_score(y, labels)

    # 2D PCA for UI plotting
    pca = PCA(n_components=2, random_state=42)
    x_pca = pca.fit_transform(x_scaled)

    return {
        "x": x_pca[:, 0].tolist(),
        "y": x_pca[:, 1].tolist(),
        "labels": labels.tolist(),
        "metrics": metrics
    }

In [ ]:
def K_means_model(x, y, max_clusters=10, user_k=None):
    # Step 1: Scale
    x_scaled, _ = standardization(x)

    # Step 2: WCSS (for elbow)
    elbow = elbow_info(x_scaled, max_clusters = max_clusters)

    # Step 3: Select k
    selected_k = user_k if user_k is not None else elbow["default_k"]

    #k = int(input("Enter the value of cluster(k): "))

    # Step 4: Final clustering
    clustering_result = fitting_K_means(x_scaled, selected_k, y)

    return {
        "elbow": elbow,
        "selected_k": selected_k,
        "clustering": clustering_result
    }

# 2). Gaussian Mixture Model (GMM)

In [ ]:
# BIC Score calculation
def gmm_model_selection(x_scaled, max_clusters=10):
    """
    Evaluate GMM for different k values.

    Decision metric:
    - BIC (minimization)
    """
    ks, bic_scores, aic_scores, silhouettes = [], [], [], []

    n_samples = x_scaled.shape[0]
    max_k = min(max_clusters, n_samples // 2)

    # computing the BIC Score for each K
    for k in range(2, max_k + 1):
        gmm = GaussianMixture(n_components=k, covariance_type="full", n_init=10, random_state=42)

        gmm.fit(x_scaled)

        ks.append(k)
        bic_scores.append(gmm.bic(x_scaled))

    # Default k chosen by BIC (statistically correct for GMM)
    default_k = ks[np.argmin(bic_scores)]

    return {
        "k": ks,
        "bic": bic_scores,
        "default_k": default_k
    }

In [ ]:
def fitting_GMM(x_scaled, k, y):
    """
    Fit final GMM model and return:
    - 2D PCA projection (for UI)
    - cluster labels
    - clustering metrics
    """

    # fitting the GMM model
    gmm_model = GaussianMixture(n_components=k, covariance_type="full", n_init=20, random_state=42)
    labels = gmm_model.fit_predict(x_scaled)

    metrics = {}

    # Silhouette valid only if more than 1 cluster
    if len(set(labels)) > 1:
        metrics["silhouette"] = silhouette_score(x_scaled, labels)


    if y is not None:
       # Adjusted Rand Index
        metrics["ari"] = adjusted_rand_score(y, labels)

    # PCA for visualization (not for clustering)
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(x_scaled)

    return {
        "x": X_pca[:, 0].tolist(),
        "y": X_pca[:, 1].tolist(),
        "labels": labels.tolist(),
        "metrics": metrics
    }

In [ ]:
def GMM_model(x, y=None, max_clusters=10, user_k=None):
    """
    Complete GMM clustering pipeline.

    Workflow:
    1. Scale data
    2. Select k using BIC
    3. Allow user override
    4. Return PCA-based visualization output
    """

    # Step 1: Standardization
    x_scaled, _ = standardization(x)

    # Step 2: Model selection
    model_info = gmm_model_selection(x_scaled, max_clusters)

    # Step 3: Choose k
    selected_k = user_k if user_k is not None else model_info["default_k"]

    # Step 4: Final clustering
    clustering_result = fitting_GMM(x_scaled, selected_k, y)

    return {
        "model_selection": model_info,
        "selected_k": selected_k,
        "clustering": clustering_result
    }

# 3). Hierarchical Clustering

In [ ]:
def dendrogram(x_scaled):
    """
    Compute linkage matrix for dendrogram visualization.

    This builds the full hierarchical tree.
    It does NOT decide the number of clusters.
    """

    # Compute linkage matrix
    Z = sch.linkage(x_scaled, method="ward")

    return Z

In [ ]:
def fitting_agglomerative(x_scaled, k, y):
    """
    Perform Agglomerative clustering using user-selected k
    and return PCA output for visualization.
    """
    # fitting the Agglomerative model
    model = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = model.fit_predict(x_scaled)

    metrics = {}

    # Silhouette valid only if more than 1 cluster
    if len(set(labels)) > 1:
        metrics["silhouette"] = silhouette_score(x_scaled, labels)


    if y is not None:
        metrics["ari"] = adjusted_rand_score(y, labels)

    # PCA only for visualization (not for clustering)
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(x_scaled)

    return {
        "x": X_pca[:, 0].tolist(),
        "y": X_pca[:, 1].tolist(),
        "labels": labels.tolist(),
        "metrics": metrics
    }

In [ ]:
def hierarchical_model(x, y=None, user_k=None):
    """
    Hierarchical clustering pipeline where:
    - dendrogram is shown to user
    - user MUST decide k
    """
    if user_k is None:
        raise ValueError(
            "user_k must be provided. "
            "Please select number of clusters from dendrogram."
        )

    # Step 1: Standardization
    x_scaled, _ = standardization(x)

    # Step 2: Get linkage matrix
    Z = dendrogram(x_scaled)

    # Step 3: Final clustering using user-selected k
    clustering_result = fitting_agglomerative(x_scaled, user_k, y)

    return {
        "selected_k": user_k,
        "clustering": clustering_result,
        "linkage_matrix": Z.tolist()
    }

# 4). DBSCAN

In [ ]:
def fitting_dbscan(x_scaled, eps, min_samples, y=None):
    dbscan = DBSCAN(
             eps=eps,          # needs tuning
             min_samples=min_samples,  # reasonable for small N
             metric="euclidean"
    )

    clusters = dbscan.fit_predict(x_scaled)

    metrics = {}

    # Silhouette only valid if more than 1 cluster and no all-noise case
    unique_labels = set(clusters)
    if len(unique_labels) > 1 and -1 not in unique_labels:
        metrics["silhouette"] = silhouette_score(x_scaled, clusters)

    if y is not None:
        metrics["ari"] = adjusted_rand_score(y, clusters)

    return clusters.tolist(), metrics

In [ ]:
def DBSCAN_model(x, y, eps=0.5, min_samples=5):
    # Step 1: Standardization
    x_scaled, _ = standardization(x)

    # Step 3: Fit DBSCAN
    clusters, metrics = fitting_dbscan(x_scaled, eps=eps, min_samples=min_samples, y=y)

    # PCA only for visualization
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(x_scaled)


    return {
        "x": X_pca[:, 0].tolist(),
        "y": X_pca[:, 1].tolist(),
        "eps": eps,
        "min_samples": min_samples,
        "clusters": clusters,
        "metrics": metrics
    }

In [ ]:
def validate_input(x):
    if not isinstance(x, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if x.empty:
        raise ValueError("Input dataset is empty.")

    if x.isnull().sum().any():
        raise ValueError("Missing values detected in dataset.")

    if not all(np.issubdtype(dtype, np.number) for dtype in x.dtypes):
        raise ValueError("All features must be numeric.")

    if len(x) < 5:
        raise ValueError("Dataset too small for clustering.")

In [ ]:
def clustering_analysis(model, X, y=None, max_clusters=10, user_k=None, **kwargs):
    """
    Central clustering dispatcher.

    Parameters:
    - model: string selected by user
    - X: feature matrix
    - y: optional ground truth labels
    - max_clusters: upper bound for cluster search
    - user_k: optional user-selected k (for KMeans/GMM)
    - kwargs: algorithm-specific parameters
    """
    # Validate Input
    validate_input(X)

    # validate y
    if y is not None:
        if len(y) != len(X):
            raise ValueError("Length of y must match number of samples in X.")


    if model == "KMeans Clustering":
        return K_means_model(x=X, y=y, max_clusters=max_clusters, user_k=user_k)
    elif model == "GMM Clustering":
        return GMM_model(x=X, y=y, max_clusters=max_clusters, user_k=user_k)
    elif model == "Hierarchical Clustering":
        return hierarchical_model(x=X, y=y, user_k=user_k)
    elif model == "DBSCAN Clustering":
        return DBSCAN_model(x=X, y=y, eps=kwargs.get("eps", 1.3), min_samples=kwargs.get("min_samples", 3))
    else:
        print("Invalid model")

In [ ]:
model = input("Enter the model: ")
clustering_analysis(model, X=X, y=y, max_clusters=10, user_k=None)

Enter the model: GMM Clustering


{'model_selection': {'k': [2, 3, 4, 5, 6, 7],
  'bic': [np.float64(-55.04696300161228),
   np.float64(-142.6215820241665),
   np.float64(-179.45212117350167),
   np.float64(-253.479827755183),
   np.float64(-286.23326157695647),
   np.float64(-234.72851286130538)],
  'default_k': 6},
 'selected_k': 6,
 'clustering': {'x': [2.6585384445536406,
   2.433573202879241,
   -2.29402013381044,
   2.8518179488403597,
   -2.4136975647795773,
   1.851429024004519,
   -2.383882650891135,
   -2.264520581824131,
   2.588696589628737,
   -2.886676855445228,
   2.3830137428218383,
   2.5136156301051273,
   -1.9636489113900484,
   -3.0742378846929026],
  'y': [-0.37989095200936257,
   -0.3283876752447488,
   -0.3325305661422246,
   -0.47344969020258115,
   -0.1600044592012271,
   2.4056794316959578,
   -0.005398065412061447,
   -0.5852314414555391,
   0.036508469917530366,
   0.49022232291589735,
   -0.31321038236309656,
   -0.4708430766653702,
   -0.15581511399437456,
   0.27235119816120096],
  'label

In [ ]:
model = input("Enter the model: ")
clustering_analysis(model, X=X, y=y, max_clusters=10, user_k=None)

Enter the model: GMM Clustering


{'model_selection': {'k': [2, 3, 4, 5, 6, 7],
  'bic': [np.float64(-55.04696300161228),
   np.float64(-142.6215820241665),
   np.float64(-179.45212117350167),
   np.float64(-253.479827755183),
   np.float64(-286.23326157695647),
   np.float64(-234.72851286130538)],
  'default_k': 6},
 'selected_k': 6,
 'clustering': {'x': [2.6585384445536406,
   2.433573202879241,
   -2.29402013381044,
   2.8518179488403597,
   -2.4136975647795773,
   1.851429024004519,
   -2.383882650891135,
   -2.264520581824131,
   2.588696589628737,
   -2.886676855445228,
   2.3830137428218383,
   2.5136156301051273,
   -1.9636489113900484,
   -3.0742378846929026],
  'y': [-0.37989095200936257,
   -0.3283876752447488,
   -0.3325305661422246,
   -0.47344969020258115,
   -0.1600044592012271,
   2.4056794316959578,
   -0.005398065412061447,
   -0.5852314414555391,
   0.036508469917530366,
   0.49022232291589735,
   -0.31321038236309656,
   -0.4708430766653702,
   -0.15581511399437456,
   0.27235119816120096],
  'label

In [ ]:
model = input("Enter the model: ")
clustering_analysis(model, X=X, y=y, user_k=2)

Enter the model: Hierarchical Clustering


{'selected_k': 2,
 'clustering': {'x': [2.6585384445536406,
   2.433573202879241,
   -2.29402013381044,
   2.8518179488403597,
   -2.4136975647795773,
   1.851429024004519,
   -2.383882650891135,
   -2.264520581824131,
   2.588696589628737,
   -2.886676855445228,
   2.3830137428218383,
   2.5136156301051273,
   -1.9636489113900484,
   -3.0742378846929026],
  'y': [-0.37989095200936257,
   -0.3283876752447488,
   -0.3325305661422246,
   -0.47344969020258115,
   -0.1600044592012271,
   2.4056794316959578,
   -0.005398065412061447,
   -0.5852314414555391,
   0.036508469917530366,
   0.49022232291589735,
   -0.31321038236309656,
   -0.4708430766653702,
   -0.15581511399437456,
   0.27235119816120096],
  'labels': [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1],
  'metrics': {'silhouette': np.float64(0.7546440860425349), 'ari': 1.0}},
 'linkage_matrix': [[1.0, 10.0, 0.46797325807412044, 2.0],
  [2.0, 7.0, 0.4706168532553468, 2.0],
  [0.0, 3.0, 0.4890474038544364, 2.0],
  [4.0, 6.0, 0.60771510951

In [ ]:
model = input("Enter the model: ")
clustering_analysis(model, X=X, y=y)

Enter the model: DBSCAN Clustering


{'x': [2.6585384445536406,
  2.433573202879241,
  -2.29402013381044,
  2.8518179488403597,
  -2.4136975647795773,
  1.851429024004519,
  -2.383882650891135,
  -2.264520581824131,
  2.588696589628737,
  -2.886676855445228,
  2.3830137428218383,
  2.5136156301051273,
  -1.9636489113900484,
  -3.0742378846929026],
 'y': [-0.37989095200936257,
  -0.3283876752447488,
  -0.3325305661422246,
  -0.47344969020258115,
  -0.1600044592012271,
  2.4056794316959578,
  -0.005398065412061447,
  -0.5852314414555391,
  0.036508469917530366,
  0.49022232291589735,
  -0.31321038236309656,
  -0.4708430766653702,
  -0.15581511399437456,
  0.27235119816120096],
 'eps': 1.3,
 'min_samples': 3,
 'clusters': [0, 0, 1, 0, 1, -1, 1, 1, 0, 1, 0, 0, 1, 1],
 'metrics': {'ari': 0.865979381443299}}

# Vizualization

1) Bar Chart

In [ ]:
def bar_chart_plot(df, label_col, label_map=None, round_digits=4):

    # dataframe validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    if label_col not in df.columns:
        raise ValueError(f"{label_col} not found in dataframe")

    # automatically select numeric features
    feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if label_col in feature_cols:
        feature_cols.remove(label_col)

    if len(feature_cols) == 0:
        raise ValueError("No numeric feature columns found.")

    # missing value check
    if df[feature_cols].isnull().any().any():
        raise ValueError("Missing values detected in numeric features.")

    df_copy = df.copy()

    # handle label mapping
    if label_map:
        df_copy["Group_label"] = df_copy[label_col].map(label_map)
    else:
        df_copy["Group_label"] = df_copy[label_col].astype(str)

    # compute group means
    means = (
        df_copy.groupby("Group_label")[feature_cols]
        .mean()
        .reset_index()
    )

    # convert to long format
    means_long = means.melt(
        id_vars="Group_label",
        value_vars=feature_cols,
        var_name="Feature",
        value_name="Mean Value"
    )

    # rounding for frontend
    means_long["Mean Value"] = means_long["Mean Value"].round(round_digits)

    return {
        "chart_type": "grouped_bar",
        "data": means_long.to_dict(orient="records")
    }

In [ ]:
bar_chart_plot(df=df, label_col="Group")

{'chart_type': 'grouped_bar',
 'data': [{'Group_label': '0',
   'Feature': 'Permeability',
   'Mean Value': 71.2857},
  {'Group_label': '1', 'Feature': 'Permeability', 'Mean Value': 20.4286},
  {'Group_label': '0', 'Feature': 'LDL_uptake', 'Mean Value': 0.5286},
  {'Group_label': '1', 'Feature': 'LDL_uptake', 'Mean Value': 1.3857},
  {'Group_label': '0', 'Feature': 'Total_ROS', 'Mean Value': 1.9571},
  {'Group_label': '1', 'Feature': 'Total_ROS', 'Mean Value': 0.9429},
  {'Group_label': '0', 'Feature': 'Vascular_Marker', 'Mean Value': 73.4286},
  {'Group_label': '1', 'Feature': 'Vascular_Marker', 'Mean Value': 96.0},
  {'Group_label': '0', 'Feature': 'Cell_Signalling', 'Mean Value': 49.4286},
  {'Group_label': '1', 'Feature': 'Cell_Signalling', 'Mean Value': 86.7143},
  {'Group_label': '0', 'Feature': 'Tube_formation', 'Mean Value': 39.5714},
  {'Group_label': '1', 'Feature': 'Tube_formation', 'Mean Value': 96.0},
  {'Group_label': '0', 'Feature': 'In_vivo_recovery', 'Mean Value': 27.8

2) Parallel plot

In [ ]:
def get_parallel_data(df, label_col="Group"):

    # Only feature columns
    feature_cols = df.select_dtypes(include="number").columns.tolist()
    feature_cols.remove(label_col)

    return {
        "chart_type": "parallel",
        "dimensions": feature_cols,
        "data": df[feature_cols].values.tolist(),
        "group": df[label_col].tolist()
    }

In [ ]:
get_parallel_data(df, label_col="Group")

{'chart_type': 'parallel',
 'dimensions': ['Permeability',
  'LDL_uptake',
  'Total_ROS',
  'Vascular_Marker',
  'Cell_Signalling',
  'Tube_formation',
  'In_vivo_recovery'],
 'data': [[71.0, 0.3, 1.8, 68.0, 43.0, 39.0, 35.0],
  [68.0, 0.34, 1.9, 75.0, 47.0, 35.0, 34.0],
  [20.0, 1.2, 1.0, 100.0, 80.0, 100.0, 70.0],
  [79.0, 0.19, 1.8, 70.0, 43.0, 41.0, 28.0],
  [17.0, 1.3, 0.9, 96.0, 88.0, 92.0, 72.0],
  [69.0, 2.0, 2.2, 74.0, 55.0, 44.0, 19.0],
  [22.0, 1.4, 1.1, 96.0, 85.0, 99.0, 79.0],
  [23.0, 1.1, 0.9, 98.0, 78.0, 98.0, 78.0],
  [70.0, 0.35, 2.1, 69.0, 59.0, 46.0, 22.0],
  [21.0, 1.8, 0.8, 92.0, 95.0, 96.0, 82.0],
  [67.0, 0.3, 1.9, 77.0, 50.0, 40.0, 25.0],
  [75.0, 0.22, 2.0, 81.0, 49.0, 32.0, 32.0],
  [27.0, 1.2, 1.2, 90.0, 89.0, 90.0, 83.0],
  [13.0, 1.7, 0.7, 100.0, 92.0, 97.0, 71.0]],
 'group': [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1]}

3) PCA Plot

In [ ]:
def pca_plot(df, label_col="Group", n_components=3):
    if n_components not in [2, 3]:
        raise ValueError("n_components must be 2 or 3")

    # Separate features and label
    X = df.drop(columns=[label_col])
    y = df[label_col]

    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Apply PCA
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)

    # Explained variance in percentage
    explained = (pca.explained_variance_ratio_ * 100).tolist()

    # Column-wise PCA coordinates for plotting
    points = {
        "pc1": X_pca[:, 0].astype(float).tolist(),
        "pc2": X_pca[:, 1].astype(float).tolist(),
        "group": y.astype(int).tolist()
    }

    if n_components == 3:
        points["pc3"] = X_pca[:, 2].astype(float).tolist()

    # Feature loadings for biplot
    loadings = []
    for i, feature in enumerate(X.columns):
        loading = {
            "feature": feature,
            "pc1": float(pca.components_.T[i, 0]),
            "pc2": float(pca.components_.T[i, 1])
        }
        if n_components == 3:
            loading["pc3"] = float(pca.components_.T[i, 2])
        loadings.append(loading)

    return {
        "chart_type": "pca_3d" if n_components == 3 else "pca_2d",
        "n_components": n_components,
        "explained_variance_percent": explained,
        "points": points,
        "loadings": loadings
    }

In [ ]:
pca_plot(df, label_col="Group", n_components=2)

{'chart_type': 'pca_2d',
 'n_components': 2,
 'explained_variance_percent': [88.55949128638862, 7.553270394983713],
 'points': {'pc1': [2.6585384445536406,
   2.433573202879241,
   -2.29402013381044,
   2.8518179488403597,
   -2.4136975647795773,
   1.851429024004519,
   -2.383882650891135,
   -2.264520581824131,
   2.588696589628737,
   -2.886676855445228,
   2.3830137428218383,
   2.5136156301051273,
   -1.9636489113900484,
   -3.0742378846929026],
  'pc2': [-0.37989095200936257,
   -0.3283876752447488,
   -0.3325305661422246,
   -0.47344969020258115,
   -0.1600044592012271,
   2.4056794316959578,
   -0.005398065412061447,
   -0.5852314414555391,
   0.036508469917530366,
   0.49022232291589735,
   -0.31321038236309656,
   -0.4708430766653702,
   -0.15581511399437456,
   0.27235119816120096],
  'group': [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1]},
 'loadings': [{'feature': 'Permeability',
   'pc1': 0.39899903565752076,
   'pc2': 0.06412796620344198},
  {'feature': 'LDL_uptake',
   'pc

4). Scatter plot

In [ ]:
def scatter_plot(df, label_col="Group", feature_x=None, feature_y=None, feature_z=None):

    if feature_x is None or feature_y is None:
        raise ValueError("feature_x and feature_y are required")

    if feature_x not in df.columns or feature_y not in df.columns:
        raise ValueError("Selected features not found in dataframe")

    if feature_z and feature_z not in df.columns:
        raise ValueError("feature_z not found in dataframe")

    chart_type = "scatter_3d" if feature_z else "scatter_2d"

    points = {
        feature_x: df[feature_x].astype(float).tolist(),
        feature_y: df[feature_y].astype(float).tolist(),
        "group": df[label_col].astype(int).tolist()
    }

    if feature_z:
        points[feature_z] = df[feature_z].astype(float).tolist()

    return {
        "chart_type": chart_type,
        "features": {
            "x": feature_x,
            "y": feature_y,
            "z": feature_z
        },
        "points": points
    }

In [ ]:
df.columns

Index(['Permeability', 'LDL_uptake', 'Total_ROS', 'Vascular_Marker',
       'Cell_Signalling', 'Tube_formation', 'In_vivo_recovery', 'Group'],
      dtype='object')

In [ ]:
scatter_plot(df, label_col="Group", feature_x="Permeability", feature_y="LDL_uptake", feature_z="Total_ROS")

{'chart_type': 'scatter_3d',
 'features': {'x': 'Permeability', 'y': 'LDL_uptake', 'z': 'Total_ROS'},
 'points': {'Permeability': [71.0,
   68.0,
   20.0,
   79.0,
   17.0,
   69.0,
   22.0,
   23.0,
   70.0,
   21.0,
   67.0,
   75.0,
   27.0,
   13.0],
  'LDL_uptake': [0.3,
   0.34,
   1.2,
   0.19,
   1.3,
   2.0,
   1.4,
   1.1,
   0.35,
   1.8,
   0.3,
   0.22,
   1.2,
   1.7],
  'group': [0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1],
  'Total_ROS': [1.8,
   1.9,
   1.0,
   1.8,
   0.9,
   2.2,
   1.1,
   0.9,
   2.1,
   0.8,
   1.9,
   2.0,
   1.2,
   0.7]}}

5). Histogram Plot


In [ ]:
def histogram_plot(df, feature, label_col="Group"):
    if feature not in df.columns:
        raise ValueError(f"{feature} not found in dataframe")

    if label_col not in df.columns:
        raise ValueError(f"{label_col} not found in dataframe")

    grouped = (
        df.groupby(label_col)[feature]
        .apply(lambda x: x.astype(float).tolist())
        .to_dict()
    )

    return {
        "chart_type": "histogram",
        "feature": feature,
        "groups": grouped
    }

In [ ]:
histogram_plot(df, feature="Permeability", label_col="Group")

{'chart_type': 'histogram',
 'feature': 'Permeability',
 'groups': {0: [71.0, 68.0, 79.0, 69.0, 70.0, 67.0, 75.0],
  1: [20.0, 17.0, 22.0, 23.0, 21.0, 27.0, 13.0]}}

In [ ]:
def visualization(df, plot_name, label_col=None, **kwargs):

    # basic validation
    if not isinstance(df, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if df.empty:
        raise ValueError("Dataset is empty.")

    if plot_name == "histogram":
        return histogram_plot(df, feature=kwargs.get("feature"), label_col=label_col)

    elif plot_name == "scatter":
        return scatter_plot(df, label_col=label_col, feature_x=kwargs.get("feature_x"), feature_y=kwargs.get("feature_y"), feature_z=kwargs.get("feature_z"))

    elif plot_name == "pca_plot":
        return pca_plot(df, label_col=label_col, n_components=kwargs.get("n_components", 2))

    elif plot_name == "bar_plot":
        return bar_chart_plot(df, label_col=label_col)

    elif plot_name == "parallel_plot":
        return get_parallel_data(df, label_col=label_col)

    else:
        raise ValueError(f"Invalid plot name: {plot_name}")

In [ ]:
def analyze_data(df=None, module=None, method=None, label_col=None, **kwargs):
    """
    Master dispatcher for the analytics engine.
    """

    # Basic validation
    if module is None:
        raise ValueError("module must be specified.")

    # Statistical Analysis
    if module == "statistics":
        return statistical_modelliing(
            df,
            test_name=method,
            label_col=label_col,
            **kwargs
        )

    # Anomaly Detection
    elif module == "anomaly":
        return ananomaly_detection(
            df,
            test_name=method,
            label_col=label_col,
            **kwargs
        )

    # Classification Models
    elif module == "classification":
        return classification_model(
            df,
            model_name=method,
            label_col=label_col,
            **kwargs
        )

    # Clustering
    elif module == "clustering":
        return clustering_analysis(
            model=method,
            X=kwargs.get("X"),
            y=kwargs.get("y"),
            max_clusters=kwargs.get("max_clusters", 10),
            user_k=kwargs.get("user_k"),
            **kwargs
        )

    # Visualization
    elif module == "visualization":
        return visualization(
            df,
            plot_name=method,
            label_col=label_col,
            **kwargs
        )

    else:
        raise ValueError(
            "Invalid module. Choose from: "
            "'statistics', 'anomaly', 'classification', "
            "'clustering', 'visualization'"
        )